# MLP Output Decomposition

Decompose MLP outputs into interpretable parts: per-neuron output directions,
position contributions, selectivity profiles, direction clustering, and
residual alignment.

## Why This Matters

Composition mlp output decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_output_decomposition import (
    per_neuron_output_direction, position_neuron_contributions,
    neuron_selectivity_profile, mlp_output_direction_clustering,
    mlp_residual_alignment,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Per-Neuron Output Direction

What does each neuron's output direction do to the vocabulary?

In [ ]:
result = per_neuron_output_direction(model, tokens, layer=0, top_k=10)
print(f"Top {result['n_analyzed']} neurons by activation:\n")
for n in result['per_neuron']:
    print(f"  Neuron {n['neuron_idx']:3d}: act={n['mean_activation']:.4f}, "
          f"dir_norm={n['output_direction_norm']:.4f}, "
          f"promotes={n['top_promoted_token']}, suppresses={n['top_suppressed_token']}")

## Position-Specific Contributions

Which neurons contribute most at the last position?

In [ ]:
result = position_neuron_contributions(model, tokens, layer=0, position=-1, top_k=5)
print(f"Total contribution: {result['total_contribution']:.4f}\n")
for n in result['per_neuron']:
    print(f"  Neuron {n['neuron_idx']:3d}: act={n['activation']:+.4f}, "
          f"contrib={n['contribution']:.4f} ({n['fraction']:.1%})")

## Neuron Selectivity

Which neurons are position-selective vs broadly active?

In [ ]:
result = neuron_selectivity_profile(model, tokens, layer=0, top_k=10)
print(f"Selective neurons: {result['n_selective']}/{len(result['per_neuron'])}\n")
for n in result['per_neuron']:
    sel_type = 'SELECTIVE' if n['is_selective'] else 'broad'
    print(f"  Neuron {n['neuron_idx']:3d}: selectivity={n['selectivity']:.3f} [{sel_type}] "
          f"peak_pos={n['peak_position']}")

## Output Direction Clustering

Do neurons cluster into groups with similar output directions?

In [ ]:
result = mlp_output_direction_clustering(model, tokens, layer=0, n_clusters=3)
print(f"Active neurons: {result['n_active']}\n")
for c in result['clusters']:
    print(f"  Cluster {c['cluster']}: {c['n_members']} neurons, "
          f"mean_act={c['mean_activation']:.4f}")

## Residual Alignment

Does the MLP reinforce or oppose the existing residual stream?

In [ ]:
result = mlp_residual_alignment(model, tokens, layer=0)
print(f"Mean alignment: {result['mean_alignment']:.4f}")
print(f"Reinforcement fraction: {result['reinforcement_fraction']:.2%}\n")
for p in result['per_position']:
    role = 'reinforces' if p['reinforces'] else 'opposes'
    print(f"  Pos {p['position']}: cos={p['cosine_alignment']:+.4f} [{role}], "
          f"resid={p['residual_norm']:.4f}, mlp={p['mlp_output_norm']:.4f}")